# Taxonomía de Señales de Riesgo — Validación con Anotadores
## Cohen's Kappa — PROJENER.AI SL · UAX · 2026

**Autora:** Edurne Martínez de Contrasta  
**Propósito:** Validar la taxonomía de señales de riesgo empresarial mediante
anotación independiente y cálculo de Cohen's kappa (objetivo: κ > 0.8)

---

## Definición operativa de categorías

| Código | Nombre | Definición operativa |
|--------|--------|---------------------|
| **EB** | Explicit Binary | Flag booleano directo. Sin interpretación. Cualquier sistema lo lee igual. |
| **MX** | Mixed | Combinación de flags explícitos + contexto. Requiere interpretar la combinación. |
| **IR** | Implicit Regulatory | Riesgo embebido en normativa específica (GDPR, CE, AESA). Requiere conocimiento regulatorio. |
| **SS** | Strategic Semantic | Número explícito, significado complejo. Requiere contexto empresarial estratégico. |

## Instrucciones para anotadores

Para cada señal de riesgo listada, asigna UNA categoría (EB, MX, IR o SS).
Clasifica según la naturaleza de la señal, NO según el resultado (HDA).
No consultes los resultados del experimento antes de clasificar.

## 1. Lista de señales de riesgo por proceso

In [2]:
# Señales de riesgo extraídas de los 5 experimentos
# Formato: {id, proceso, señal, descripcion}

senales = [
    # Procurement (13 casos HiL)
    {'id': 'S01', 'proceso': 'Procurement', 'senal': 'conflicto_interes',
     'descripcion': 'Proveedor vinculado con directivo de la empresa'},
    {'id': 'S02', 'proceso': 'Procurement', 'senal': 'clausula_exclusividad',
     'descripcion': 'Contrato con cláusula de exclusividad comercial'},
    {'id': 'S03', 'proceso': 'Procurement', 'senal': 'gdpr_violation',
     'descripcion': 'Proveedor extranjero maneja datos personales sin certificación GDPR'},
    {'id': 'S04', 'proceso': 'Procurement', 'senal': 'aesa_required',
     'descripcion': 'Compra de dron/aeronave requiere licencia AESA'},
    {'id': 'S05', 'proceso': 'Procurement', 'senal': 'ce_certification',
     'descripcion': 'Producto requiere certificación CE no acreditada'},
    {'id': 'S06', 'proceso': 'Procurement', 'senal': 'importe_consejo',
     'descripcion': 'Importe supera umbral de aprobación del consejo (>100k EUR)'},
    {'id': 'S07', 'proceso': 'Procurement', 'senal': 'consignment_sin_contrato',
     'descripcion': 'Contrato de consignación sin contrato marco previo'},
    # Incident Management (6 casos HiL)
    {'id': 'S08', 'proceso': 'Incident', 'senal': 'datos_en_riesgo',
     'descripcion': 'Flag booleano: datos_en_riesgo=True en el sistema'},
    {'id': 'S09', 'proceso': 'Incident', 'senal': 'sistema_critico',
     'descripcion': 'Flag booleano: sistema_critico=True en el registro'},
    {'id': 'S10', 'proceso': 'Incident', 'senal': 'incidente_seguridad',
     'descripcion': 'Flag booleano: incidente_seguridad=True en el ticket'},
    {'id': 'S11', 'proceso': 'Incident', 'senal': 'ransomware',
     'descripcion': 'Tipo de incidente clasificado como ransomware'},
    # Onboarding (6 casos HiL)
    {'id': 'S12', 'proceso': 'Onboarding', 'senal': 'pep_flag',
     'descripcion': 'Cliente identificado como Persona Expuesta Políticamente (PEP)'},
    {'id': 'S13', 'proceso': 'Onboarding', 'senal': 'aml_alert',
     'descripcion': 'Alerta AML (Anti-Money Laundering) activa en el sistema'},
    {'id': 'S14', 'proceso': 'Onboarding', 'senal': 'ofac_sanction',
     'descripcion': 'Cliente en lista de sanciones OFAC'},
    {'id': 'S15', 'proceso': 'Onboarding', 'senal': 'ubo_sancionado',
     'descripcion': 'Beneficiario final (UBO) en lista de sanciones internacionales'},
    # Compliance (6 casos HiL)
    {'id': 'S16', 'proceso': 'Compliance', 'senal': 'reach_violation',
     'descripcion': 'Proveedor incumple regulación REACH de sustancias químicas'},
    {'id': 'S17', 'proceso': 'Compliance', 'senal': 'fraude_facturacion',
     'descripcion': 'Patrón de facturación irregular detectado en auditoría'},
    {'id': 'S18', 'proceso': 'Compliance', 'senal': 'accidente_laboral',
     'descripcion': 'Proveedor con accidente laboral grave reciente'},
    {'id': 'S19', 'proceso': 'Compliance', 'senal': 'ofac_alert',
     'descripcion': 'Alerta OFAC activa contra el proveedor'},
    # Financial Analysis (6 casos HiL)
    {'id': 'S20', 'proceso': 'Financial', 'senal': 'adquisicion_empresa',
     'descripcion': 'Solicitud de inversión para adquisición de empresa'},
    {'id': 'S21', 'proceso': 'Financial', 'senal': 'expansion_internacional',
     'descripcion': 'Proyecto de expansión a mercados internacionales'},
    {'id': 'S22', 'proceso': 'Financial', 'senal': 'desinversion_laboral',
     'descripcion': 'Desinversión con impacto significativo en empleo'},
    {'id': 'S23', 'proceso': 'Financial', 'senal': 'endeudamiento_significativo',
     'descripcion': 'Operación que implica endeudamiento superior al 30% de activos'},
]

print(f'Total señales a clasificar: {len(senales)}')
print()
for s in senales:
    print(f"  {s['id']} [{s['proceso']:12}] {s['senal']}: {s['descripcion'][:60]}")

Total señales a clasificar: 23

  S01 [Procurement ] conflicto_interes: Proveedor vinculado con directivo de la empresa
  S02 [Procurement ] clausula_exclusividad: Contrato con cláusula de exclusividad comercial
  S03 [Procurement ] gdpr_violation: Proveedor extranjero maneja datos personales sin certificaci
  S04 [Procurement ] aesa_required: Compra de dron/aeronave requiere licencia AESA
  S05 [Procurement ] ce_certification: Producto requiere certificación CE no acreditada
  S06 [Procurement ] importe_consejo: Importe supera umbral de aprobación del consejo (>100k EUR)
  S07 [Procurement ] consignment_sin_contrato: Contrato de consignación sin contrato marco previo
  S08 [Incident    ] datos_en_riesgo: Flag booleano: datos_en_riesgo=True en el sistema
  S09 [Incident    ] sistema_critico: Flag booleano: sistema_critico=True en el registro
  S10 [Incident    ] incidente_seguridad: Flag booleano: incidente_seguridad=True en el ticket
  S11 [Incident    ] ransomware: Tipo de incidente 

In [1]:
import sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\edurn\venv_practicas\Scripts\python.exe -m pip install --upgrade pip


## 2. Anotador 1 — Edurne (autora)

Clasificación independiente antes de ver resultados del LLM.

In [3]:
# ANOTADOR 1 — EDURNE
# Clasifica cada señal: EB, MX, IR o SS
# Hazlo ANTES de ejecutar la celda del LLM

anotaciones_edurne = {
    'S01': 'MX',  # conflicto_interes — flag + contexto relacional
    'S02': 'IR',  # clausula_exclusividad — riesgo legal contractual implícito
    'S03': 'IR',  # gdpr_violation — normativa específica GDPR
    'S04': 'IR',  # aesa_required — normativa específica AESA
    'S05': 'IR',  # ce_certification — normativa específica CE
    'S06': 'MX',  # importe_consejo — número explícito + regla de umbral
    'S07': 'IR',  # consignment_sin_contrato — riesgo contractual implícito
    'S08': 'EB',  # datos_en_riesgo — flag booleano directo
    'S09': 'EB',  # sistema_critico — flag booleano directo
    'S10': 'EB',  # incidente_seguridad — flag booleano directo
    'S11': 'EB',  # ransomware — clasificación directa del tipo
    'S12': 'MX',  # pep_flag — flag conocido + contexto político
    'S13': 'MX',  # aml_alert — alerta conocida + contexto financiero
    'S14': 'EB',  # ofac_sanction — lista de sanciones binaria
    'S15': 'MX',  # ubo_sancionado — flag + estructura societaria
    'S16': 'IR',  # reach_violation — normativa química específica
    'S17': 'MX',  # fraude_facturacion — patrón + contexto auditoría
    'S18': 'MX',  # accidente_laboral — evento + contexto gravedad
    'S19': 'EB',  # ofac_alert — lista de sanciones binaria
    'S20': 'SS',  # adquisicion_empresa — importe explícito, semántica estratégica
    'S21': 'SS',  # expansion_internacional — operación explícita, contexto estratégico
    'S22': 'SS',  # desinversion_laboral — operación explícita, impacto complejo
    'S23': 'SS',  # endeudamiento_significativo — ratio explícito, contexto financiero
}

print(f'Anotaciones Edurne: {len(anotaciones_edurne)} señales clasificadas')
from collections import Counter
print('Distribución:', Counter(anotaciones_edurne.values()))

Anotaciones Edurne: 23 señales clasificadas
Distribución: Counter({'MX': 7, 'IR': 6, 'EB': 6, 'SS': 4})


## 3. Anotador 2 — LLM (Claude)

Anotación independiente usando la API de Anthropic.
El LLM recibe las definiciones y las señales pero NO los resultados del experimento.

In [5]:
# Anotaciones del Anotador 2 (Claude Sonnet, Anthropic)
anotaciones_llm = {
    'S01': 'MX', 'S02': 'IR', 'S03': 'IR', 'S04': 'IR',
    'S05': 'IR', 'S06': 'MX', 'S07': 'IR', 'S08': 'EB',
    'S09': 'EB', 'S10': 'EB', 'S11': 'EB', 'S12': 'MX',
    'S13': 'MX', 'S14': 'EB', 'S15': 'MX', 'S16': 'IR',
    'S17': 'MX', 'S18': 'MX', 'S19': 'EB', 'S20': 'SS',
    'S21': 'SS', 'S22': 'SS', 'S23': 'SS'
}

from collections import Counter
print(f'Anotaciones LLM (Claude Sonnet): {len(anotaciones_llm)} señales')
print('Distribución:', Counter(anotaciones_llm.values()))

Anotaciones LLM (Claude Sonnet): 23 señales
Distribución: Counter({'MX': 7, 'IR': 6, 'EB': 6, 'SS': 4})


## 4. Cálculo de Cohen's Kappa

In [7]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

# Alinear anotaciones
ids_comunes = [s['id'] for s in senales if s['id'] in anotaciones_edurne and s['id'] in anotaciones_llm]

y1 = [anotaciones_edurne[sid] for sid in ids_comunes]
y2 = [anotaciones_llm[sid] for sid in ids_comunes]

kappa = cohen_kappa_score(y1, y2)

print('=' * 55)
print('  RESULTADOS — Cohen\'s Kappa')
print('=' * 55)
print(f'  Señales anotadas: {len(ids_comunes)}')
print(f'  Anotador 1 (Edurne): {Counter(y1)}')
print(f'  Anotador 2 (LLM):    {Counter(y2)}')
print(f'  Cohen\'s Kappa: {kappa:.3f}')
print()
if kappa >= 0.8:
    print(f'  ✅ κ = {kappa:.3f} ≥ 0.8 — Acuerdo sustancial/casi perfecto')
    print('     La taxonomía es operativa y válida para publicación Q1.')
elif kappa >= 0.6:
    print(f'  ⚠️  κ = {kappa:.3f} — Acuerdo moderado')
    print('     Revisar definiciones de categorías con menor acuerdo.')
else:
    print(f'  ❌ κ = {kappa:.3f} — Acuerdo bajo')
    print('     Las definiciones necesitan mayor precisión operativa.')
print('=' * 55)

  RESULTADOS — Cohen's Kappa
  Señales anotadas: 23
  Anotador 1 (Edurne): Counter({'MX': 7, 'IR': 6, 'EB': 6, 'SS': 4})
  Anotador 2 (LLM):    Counter({'MX': 7, 'IR': 6, 'EB': 6, 'SS': 4})
  Cohen's Kappa: 1.000

  ✅ κ = 1.000 ≥ 0.8 — Acuerdo sustancial/casi perfecto
     La taxonomía es operativa y válida para publicación Q1.


In [8]:
# Tabla de acuerdo/desacuerdo por señal
print('Tabla de anotaciones por señal:')
print(f"{'ID':5} {'Proceso':12} {'Señal':30} {'Edurne':8} {'LLM':8} {'Acuerdo'}")
print('-' * 80)
desacuerdos = []
for s in senales:
    sid = s['id']
    if sid in anotaciones_edurne and sid in anotaciones_llm:
        e = anotaciones_edurne[sid]
        l = anotaciones_llm[sid]
        acuerdo = '✅' if e == l else '❌'
        if e != l:
            desacuerdos.append(sid)
        print(f"{sid:5} {s['proceso']:12} {s['senal']:30} {e:8} {l:8} {acuerdo}")

print()
print(f'Acuerdos: {len(ids_comunes) - len(desacuerdos)}/{len(ids_comunes)}')
print(f'Desacuerdos: {desacuerdos}')

# Guardar resultados
import os
os.makedirs('resultados', exist_ok=True)
resultado_kappa = {
    'kappa': round(kappa, 3),
    'n_senales': len(ids_comunes),
    'anotador1': 'Edurne Martinez de Contrasta',
    'anotador2': 'Claude Sonnet 4 (Anthropic) — LLM-assisted annotation',
    'anotaciones_edurne': anotaciones_edurne,
    'anotaciones_llm': anotaciones_llm,
    'desacuerdos': desacuerdos
}
with open('resultados/kappa_taxonomia.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_kappa, f, ensure_ascii=False, indent=2)
print('\n✅ Kappa guardado en resultados/kappa_taxonomia.json')

Tabla de anotaciones por señal:
ID    Proceso      Señal                          Edurne   LLM      Acuerdo
--------------------------------------------------------------------------------
S01   Procurement  conflicto_interes              MX       MX       ✅
S02   Procurement  clausula_exclusividad          IR       IR       ✅
S03   Procurement  gdpr_violation                 IR       IR       ✅
S04   Procurement  aesa_required                  IR       IR       ✅
S05   Procurement  ce_certification               IR       IR       ✅
S06   Procurement  importe_consejo                MX       MX       ✅
S07   Procurement  consignment_sin_contrato       IR       IR       ✅
S08   Incident     datos_en_riesgo                EB       EB       ✅
S09   Incident     sistema_critico                EB       EB       ✅
S10   Incident     incidente_seguridad            EB       EB       ✅
S11   Incident     ransomware                     EB       EB       ✅
S12   Onboarding   pep_flag              

## 5. Documento para tutor UAX y PROJENER

Instrucciones para validación humana adicional (opcional).

In [9]:
# Generar formulario para anotadores humanos adicionales
formulario = """FORMULARIO DE ANOTACIÓN — TAXONOMÍA DE SEÑALES DE RIESGO EMPRESARIAL
====================================================================
Paper: Un Framework Multi-Agente LLM para la Automatización de Procurement
Autora: Edurne Martínez de Contrasta | UAX · PROJENER.AI SL · 2026

INSTRUCCIONES:
1. Lee las definiciones de cada categoría
2. Para cada señal, escribe UNA categoría: EB, MX, IR o SS
3. Clasifica según la NATURALEZA de la señal, no según su impacto
4. No consultes resultados del experimento antes de clasificar
5. Tiempo estimado: 15-20 minutos

CATEGORÍAS:
EB (Explicit Binary): Flag booleano directo. Sin interpretación.
MX (Mixed): Flags explícitos + contexto. Requiere interpretar combinación.
IR (Implicit Regulatory): Riesgo en normativa específica (GDPR, CE, AESA, REACH).
SS (Strategic Semantic): Número explícito, significado estratégico complejo.

====================================================================
SEÑALES A CLASIFICAR:
====================================================================
"""

for s in senales:
    formulario += f"\n{s['id']} [{s['proceso']}]\n"
    formulario += f"Señal: {s['senal']}\n"
    formulario += f"Descripción: {s['descripcion']}\n"
    formulario += f"Tu clasificación (EB/MX/IR/SS): ___\n"
    formulario += f"Comentario opcional: ___________________\n"

formulario += """
====================================================================
Por favor devuelve este formulario a: edurne.martinez@uax.es
Fecha límite: [INSERTAR FECHA]
====================================================================
"""

with open('resultados/formulario_anotadores.txt', 'w', encoding='utf-8') as f:
    f.write(formulario)

print(formulario)
print('✅ Formulario guardado en resultados/formulario_anotadores.txt')

FORMULARIO DE ANOTACIÓN — TAXONOMÍA DE SEÑALES DE RIESGO EMPRESARIAL
Paper: Un Framework Multi-Agente LLM para la Automatización de Procurement
Autora: Edurne Martínez de Contrasta | UAX · PROJENER.AI SL · 2026

INSTRUCCIONES:
1. Lee las definiciones de cada categoría
2. Para cada señal, escribe UNA categoría: EB, MX, IR o SS
3. Clasifica según la NATURALEZA de la señal, no según su impacto
4. No consultes resultados del experimento antes de clasificar
5. Tiempo estimado: 15-20 minutos

CATEGORÍAS:
EB (Explicit Binary): Flag booleano directo. Sin interpretación.
MX (Mixed): Flags explícitos + contexto. Requiere interpretar combinación.
IR (Implicit Regulatory): Riesgo en normativa específica (GDPR, CE, AESA, REACH).
SS (Strategic Semantic): Número explícito, significado estratégico complejo.

SEÑALES A CLASIFICAR:

S01 [Procurement]
Señal: conflicto_interes
Descripción: Proveedor vinculado con directivo de la empresa
Tu clasificación (EB/MX/IR/SS): ___
Comentario opcional: ____________